### Introducción

El reto supone la construcción de un autocodificador **no-determinístico convolucional**, es decir:
- Existe un espacio latente único. Por lo que, para cada entrada existe solo un punto en el espacio latente.
- No se introduce ruido ni distribuciones de probabilidad en el espacio latente.
- El autocodificador solo considera capas densas o convolucionales, entrenandas con pérdida como MSE o BCE.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf

In [2]:
# Obtención del conjunto de datos
(x_entrenamiento, y_entrenamiento), (x_prueba, y_prueba) = tf.keras.datasets.mnist.load_data()

# Normalización del conjunto de datos. Las intensidades de los pixeles se encuentran entre [0,1]
x_entrenamiento = x_entrenamiento / 255.0

# Estimación de la media y desviación estándar correspondiente al conjunto de entrenamiento.
media = np.mean(x_entrenamiento)
dvstd = np.std(x_entrenamiento)

# Normalización del conjunto de datos
x_entrenamiento = (x_entrenamiento - media) / dvstd
x_entrenamiento = np.expand_dims(x_entrenamiento, axis=3)

# Esto es solo para asegurar que los datos esten almaceados en flotantes de 32 bits
x_entrenamiento = tf.cast(x_entrenamiento, dtype=tf.float32)

# El vector de etLas etiquetas son transformadas a codificación one-hot vectors
y_entrenamiento_onehot = tf.keras.utils.to_categorical(y_entrenamiento, num_classes=10)

datos_entrenamiento = tf.data.Dataset.from_tensor_slices((x_entrenamiento, y_entrenamiento_onehot))
datos_entrenamiento = datos_entrenamiento.shuffle(1000).batch(64)

In [7]:
class CVAE(tf.keras.Model):
    def __init__(self, dim_latente):
        super().__init__()
        self.dim_latente = dim_latente

        # Fracción de la red que extrae características que son invariantes a la traslación local.
        caracteristicas = tf.keras.Input(shape=(28, 28, 1))
        x = tf.keras.layers.Conv2D(32, kernel_size=3, strides=2, activation='relu')(caracteristicas)
        x = tf.keras.layers.Conv2D(64, kernel_size=3, strides=2, activation='relu')(x)
        x = tf.keras.layers.Flatten()(x)
        self.codificador_caracteristicas = tf.keras.Model(inputs=caracteristicas, outputs=x)

        # Fracción de la red que aprende (generaliza) los parámetros de la distribución de interés 
        # a partir de la muestra. Note, cada muestra esta es analizada para estimar su media 
        # y desviación estandar. Estos parametros corresponden a la distribución continua Gaussiana.
        
        # El tamaño 2304 viene del tamaño de las anteriores capas. La lógica es como sigue:
        # 1. Conv2D aplica 32 kernels de 3x3, con stride 2 y sin padding. Esto reduce (28,28) a (13,13) 
        #    porque (28-3)/2 + 1 = 13 por dimensión. Consecuentemente, la salida es (13, 13, 32).
        # 2. Conv2D aplica 64 kernels de 3x3, con stride 2 y sin padding. Esto reduce (13,13) a (6,6)
        #    porque (13-3)/2 + 1 = 6 por dimensión. Consecuentemente, la salida es (6,6,64)=2304.
        # El valor 10 sigue de que son diez clases las que comprende el conjunto de datos.
        vector_condicional = tf.keras.Input(shape=(2304 + 10,))
        mu_t  = tf.keras.layers.Dense(units=dim_latente)(vector_condicional)
        rho_t = tf.keras.layers.Dense(units=dim_latente)(vector_condicional)
        self.codificador_condicional = tf.keras.Model(inputs=vector_condicional, outputs=[mu_t, rho_t], name="Codif Caracteristicas")

        # El codificador resulta de la concatenación de 
        # 1. La red de extracción de invariantes sobre el vector de características.
        # 2. La red de aprendizaje condicional
        imagen    = tf.keras.Input(shape=(28, 28, 1))
        etiquetas = tf.keras.Input(shape=(10,))
        invariantes = self.codificador_caracteristicas(imagen)
        concatenado = tf.keras.layers.Concatenate()([invariantes, etiquetas])
        mu, rho = self.codificador_condicional(concatenado)
        
        # Definición final del codificador
        self.codificador = tf.keras.Model(inputs=[imagen, etiquetas], outputs=[mu, rho], name="Codificador")

        # Definción de las capas del decodificador. El decodificador recibe como entrada 
        # la variable latente (z) y la etiqueta de intequeta condicional. 
        # Suponga dim_latente = 15, esto ompica que el vector de entrada es 
        # de tamaño 25 = (15 + 10)
        entrada_decod = tf.keras.Input(shape=(dim_latente + 10,))
        # El vector de entrada es dispersado a una capa densa con 1568 unidades
        # esto corresponde al producto de 7 x 7 x 32. Este número se interpreta 
        # como sigue, una imagen de tamaño 7 x 7 con 32 canales. Este valor es 
        # intensional. Puede se otro, p. ej. 7 x 7 con 16 o 64 u otro valor.
        x = tf.keras.layers.Dense(7 * 7 * 32, activation='relu')(entrada_decod)
        # los 1568 coeficientes que salen de la capa densa son transformados en 
        # 32 matrices de tamaño 7 x 7.
        x = tf.keras.layers.Reshape((7, 7, 32))(x)
        # A cada canal se le aplica 64 núcleos de 3 x 3 con stride 2. Dado que son 
        # 64 nucleos se obtienen (__, __, 64) canales. Un stride de 2 implica que el
        # tamaño de las imagenes se duplican, por lo tanto (14, 14, 64)
        x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
        # Aplicando el anterior proceso de razonamiento, la entrada a la siguiente capa
        # es (14,14,64) y la salida es (28,28,32)
        x = tf.keras.layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
        # Finalmente, el numero de canales se reduce a 1. Dado que el estride es 1,
        # el tamaño de la imagen no se duplica. Consecuentemente, (28,28,32) se transforma
        # a (28,28,1)
        decodificado = tf.keras.layers.Conv2DTranspose(1, 3, strides=1, padding='same')(x)
        self.decodificador = tf.keras.Model(inputs=entrada_decod, outputs=decodificado, name="Decodificador")

    # En Keras (y en TensorFlow), la función call() define el paso hacia adelante (forward pass).
    # El modelo procesa la entrada y produce la salida. 
    def call(self, imagen, etiqueta):
        # Aprendizaje y aplicación de la distribución normal (a.k.a Gaussiana)
        z_mu, z_rho = self.codificador([imagen, etiqueta])
        epsilon = tf.random.normal(shape=tf.shape(z_mu))
        # Aplicación de la reparametrización
        z = z_mu + tf.math.softplus(z_rho) * epsilon

        entrada_cond = tf.concat([z, etiqueta], axis=1)
        decodificado = self.decodificador(entrada_cond)
        return z_mu, z_rho, decodificado


In [8]:
def kl_loss(z_mu, z_rho):
    """Metrica KL"""
    sigma_cuadrado = tf.math.softplus(z_rho) ** 2
    kl_1d = -0.5 * (1 + tf.math.log(sigma_cuadrado) - z_mu ** 2 - sigma_cuadrado)
    kl_batch = tf.reduce_mean(tf.reduce_sum(kl_1d,axis=1))

    return kl_batch


def elbo(z_mu, z_rho, decodificado, original):
    """Perdida de reconstruccion"""
    mse = tf.reduce_mean(tf.reduce_sum(tf.square(original - decodificado),axis=1))
    kl = kl_loss(z_mu, z_rho)

    return mse, kl

In [9]:
def entrena(modelo, beta, epochs, datos):
    optimizador = tf.keras.optimizers.Adam(learning_rate=0.001)
    registro_kl  = tf.keras.metrics.Mean(name='Perdida_kl')
    registro_mse = tf.keras.metrics.Mean(name='Perdida_mse')
    
    for epoch in range(epochs):
        for _, (imgs, etiquetas) in datos.enumerate():
            with tf.GradientTape() as tape:
                z_mu, z_rho, decodificado = modelo(imgs, etiquetas)
                mse, kl = elbo(z_mu, z_rho, decodificado, imgs)
                perdida = mse + beta * kl

            gradientes = tape.gradient(perdida, modelo.variables)
            optimizador.apply_gradients(zip(gradientes, modelo.variables))

            registro_kl.update_state(beta * kl)
            registro_mse.update_state(mse)

        print(f"Epoch {epoch + 1}: MSE = {mse}, KL = {kl:.6f}")


In [10]:
dim_latente = 15
beta = 1e-11
epochs = 10

modelo = CVAE(dim_latente)
entrena(modelo, beta, epochs, datos_entrenamiento)
#generate_conditioned_digits(model, dataset_mean, dataset_std)


Epoch 1: MSE = 3.623656988143921, KL = 211.667862
Epoch 2: MSE = 2.890350341796875, KL = 175.854156
Epoch 3: MSE = 2.5904388427734375, KL = 154.399216
Epoch 4: MSE = 2.6035358905792236, KL = 145.203232
Epoch 5: MSE = 2.2552123069763184, KL = 140.621841
Epoch 6: MSE = 1.8272477388381958, KL = 136.923645
Epoch 7: MSE = 2.0545859336853027, KL = 142.670654
Epoch 8: MSE = 2.679187297821045, KL = 141.686157
Epoch 9: MSE = 2.797980785369873, KL = 143.307678
Epoch 10: MSE = 1.8981115818023682, KL = 148.105377
